# 주문(orders, order_items) CRUD + 트랜잭션

- 주문 생성은 DB 함수 `create_order`로 처리 → **주문 저장 + 주문상품 저장 + 재고 차감**이 하나의 트랜잭션
- 중간에 실패하면(재고 부족 등) 전부 ROLLBACK

In [1]:
from shop_client import new_client, login, TEST_PASSWORD

In [2]:
admin = login("admin01@test.com")
user1 = login("user01@test.com")
user2 = login("user02@test.com")

def product_id(name):
    return admin.table("products").select("id").eq("name", name).single().execute().data["id"]

def show_stock():
    res = admin.table("products").select("name, stock").order("id").execute()
    print("재고:", {p["name"]: p["stock"] for p in res.data})

mouse = product_id("무선 마우스")
keyboard = product_id("기계식 키보드")
monitor = product_id("27인치 모니터")
show_stock()

재고: {'무선 마우스': 15, '기계식 키보드': 5, '27인치 모니터': 3}


## Create: 주문 생성 (COMMIT)

In [3]:
order_id = user1.rpc("create_order", {"items": [
    {"product_id": mouse, "quantity": 2},
    {"product_id": keyboard, "quantity": 1},
]}).execute().data
print("주문 완료! 주문번호:", order_id)

show_stock()

주문 완료! 주문번호: 1
재고: {'무선 마우스': 13, '기계식 키보드': 4, '27인치 모니터': 3}


## 트랜잭션 실패 → ROLLBACK 확인
마우스는 재고 충분, 모니터는 재고 부족 → 마우스 재고도 줄면 안 됨

In [4]:
def count_orders():
    return len(admin.table("orders").select("id").execute().data)

print("--- 주문 전 ---")
show_stock()
before = count_orders()

try:
    user1.rpc("create_order", {"items": [
        {"product_id": mouse, "quantity": 1},
        {"product_id": monitor, "quantity": 100},
    ]}).execute()
except Exception as e:
    print("주문 실패:", e)

print("--- 주문 후 ---")
show_stock()
print("주문 개수:", before, "->", count_orders(), "(변화 없음 = ROLLBACK)")

--- 주문 전 ---


재고: {'무선 마우스': 13, '기계식 키보드': 4, '27인치 모니터': 3}


주문 실패: {'message': '재고 부족: 27인치 모니터 (재고 3개, 주문 100개)', 'code': 'P0001', 'hint': None, 'details': None}
--- 주문 후 ---


재고: {'무선 마우스': 13, '기계식 키보드': 4, '27인치 모니터': 3}


주문 개수: 1 -> 1 (변화 없음 = ROLLBACK)


## Read: 내 주문 조회 (주문상품 + 상품명 함께)

In [5]:
user2.rpc("create_order", {"items": [{"product_id": monitor, "quantity": 1}]}).execute()

res = user1.table("orders").select(
    "id, total_price, status, order_items(quantity, price, products(name))"
).order("id").execute()
print("user01의 주문:")
for o in res.data:
    print(" ", o)

print("user02의 주문 수:", len(user2.table("orders").select("id").execute().data))
print("관리자가 보는 전체 주문 수:", len(admin.table("orders").select("id").execute().data))

user01의 주문:
  {'id': 1, 'total_price': 133000, 'status': 'ordered', 'order_items': [{'price': 22000, 'products': {'name': '무선 마우스'}, 'quantity': 2}, {'price': 89000, 'products': {'name': '기계식 키보드'}, 'quantity': 1}]}
user02의 주문 수: 1


관리자가 보는 전체 주문 수: 2


## 예외: 다른 사람 주문은 안 보임 (RLS)

In [6]:
res = user2.table("orders").select("id").eq("id", order_id).execute()
print("user02가 user01 주문 조회:", res.data)

user02가 user01 주문 조회: []


## Update: 주문 취소 (status 변경)

In [7]:
res = user1.table("orders").update({"status": "cancelled"}).eq("id", order_id).execute()
print("변경 결과:", res.data[0]["id"], res.data[0]["status"])

변경 결과: 1 cancelled


## Delete: 주문 삭제 (order_items도 같이 삭제 - ON DELETE CASCADE)

In [8]:
print("삭제 전 주문상품:", admin.table("order_items").select("id").eq("order_id", order_id).execute().data)

user1.table("orders").delete().eq("id", order_id).execute()

print("삭제 후 주문:", admin.table("orders").select("id").eq("id", order_id).execute().data)
print("삭제 후 주문상품:", admin.table("order_items").select("id").eq("order_id", order_id).execute().data)

삭제 전 주문상품: [{'id': 1}, {'id': 2}]


삭제 후 주문: []
삭제 후 주문상품: []


## 예외: 없는 상품 / 빈 주문

In [9]:
for items in ([{"product_id": 99999, "quantity": 1}], []):
    try:
        user1.rpc("create_order", {"items": items}).execute()
    except Exception as e:
        print("주문 실패:", e)

주문 실패: {'message': '없는 상품입니다 (id=99999)', 'code': 'P0001', 'hint': None, 'details': None}


주문 실패: {'message': '주문할 상품이 없습니다', 'code': 'P0001', 'hint': None, 'details': None}
